# Notebook 02 — `forecast_only` mode: Pareto CSVs → equations → σ ladder

## Prerequisites

1. **PYTHONPATH** — `src/` of this repo + `lya_emulator_full`:
   ```
   export PYTHONPATH=/path/to/lya_emulator_full:<repo_root>/src
   ```
2. **`data/kodiaq_gp/`** — the GP emulator basedir (committed to the repo).
3. **`data/single_z_1pvar/`** — regenerated 1pvar HDF5s. Needed when
   loading Pareto CSVs (`forecast_only` uses them to reconstruct the per-
   parameter normalization). Generate with:
   ```bash
   python scripts/regen_1pvar.py --basedir data/kodiaq_gp \
       --output data/single_z_1pvar
   ```
4. **Pareto CSVs** — at least one source must be available:
   - `bundled_baseline`: vendored CSVs shipped with the repo
     (`src/priya_forecast/_vendored/data/pareto_baseline/z3.6/`).
   - `per_parameter`: paths specified explicitly in the YAML.
   - `from_refit`: output of a prior `refit_and_forecast` run.
   
   If no CSVs are available the mode still emits σ_GP and σ_perfect_1D;
   σ_PySR is marked unavailable in the scorecard.

## What this notebook does

Demonstrates the `forecast_only` mode: load Pareto CSVs, pick one
equation per parameter (Fisher-safety filter then `pick:` rule), build the
additive-Taylor combined model, and run the three Fisher forecasts:

| Label | Forward model | Meaning |
|-------|--------------|--------|
| **σ_GP** | Full GP emulator | Reference baseline |
| **σ_perfect_1D** | Additive combine with GP 1D slices (all refits=None) | Combine ceiling with exact GP responses |
| **σ_PySR** | Additive combine with fitted PySR equations | Constraint from symbolic model |

### The additive-Taylor combine formula

```
P_F(θ, k) = P_GP(θ_fid, k)
           + Σ_i [ eq_i(θ_i, k, r=0.8) − eq_i(θ_i_fid, k, r=0.8) ]
```

At θ = θ_fid every bracket is zero, so `P_F(θ_fid) ≡ P_GP(θ_fid)` exactly.

### The σ_perfect_1D ≡ σ_GP identity

When all equations are replaced by GP 1D slices (the `perfect_1D` model),
the additive combine reproduces the GP's exact first derivatives along
each parameter axis. Because Fisher depends only on first derivatives,
**σ_perfect_1D equals σ_GP up to numerical precision** for the additive
combine. The quantity to watch is `σ_PySR / σ_GP` — the information loss
from PySR fit error.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../").resolve()
LYA_EMU_ROOT = Path("/home/mfho/student_projects/lya_emulator_full")

for p in [str(REPO_ROOT / "src"), str(LYA_EMU_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("sys.path configured.")

## 1. Configure and validate

Set `pareto_csvs.source` to match the CSVs you have available.
This example uses `bundled_baseline` (vendored) so it works out of the box
once the baseline Pareto set is populated. If you are working from your own
per-parameter CSV files, change the source to `per_parameter` and fill in
the paths in `per_parameter:`.

In [ ]:
from priya_forecast.single_z.config import (
    PipelineConfig, DataConfig, GPConfig, KRange,
    ParetoCSVsConfig,
)

cfg = PipelineConfig(
    mode="forecast_only",
    redshift=3.6,
    output_dir="/tmp/nb02_forecast_only/",
    data=DataConfig(source="kodiaq", conservative=True, mock_data="gp"),
    gp=GPConfig(basedir=str(REPO_ROOT / "data/kodiaq_gp")),
    k_range=KRange(min=0.001, max=0.04),
    combine="additive",
    pick="best_loss",
    pareto_csvs=ParetoCSVsConfig(source="bundled_baseline"),
)
cfg.validate()
print(f"mode     : {cfg.mode}")
print(f"redshift : {cfg.redshift}")
print(f"combine  : {cfg.combine}")
print(f"pick     : {cfg.pick}")
print(f"source   : {cfg.pareto_csvs.source}")

## 2. Run the pipeline

`run_forecast_only` does:
1. Try to resolve Pareto CSVs via `forecast.resolve_pareto_csvs(cfg)`.
2. For each CSV: filter Fisher-pathological Pareto rows, apply `cfg.pick`,
   reconstruct a `Refit1DResult` via `forecast.build_refit_from_pareto`.
3. Build three combined models via `combine.build_combined_model`:
   - GP model (plain).
   - `perfect_1D` model (all refits=None → GP slices).
   - PySR model (refits from Pareto CSVs).
4. Run `fisher_matrix` for all three; write outputs.

If Pareto CSVs are missing (FileNotFoundError), it falls back to
emitting σ_GP and σ_perfect_1D only.

Expected runtime: similar to `gp_only` plus three Fisher runs (~3–8 min).

In [ ]:
from priya_forecast.single_z.pipeline import run

result = run(cfg)

print("Result keys:", list(result.keys()))
print("PySR available:", result["pysr_available"])

## 3. Inspect the σ ladder

In [ ]:
import numpy as np

sigmas = result["sigmas"]    # dict: "GP" | "perfect_1D" | "PySR" → ndarray
params = cfg.parameters

print(f"{'param':<12s}  {'σ_GP':>10s}  {'σ_perf1D':>10s}  {'σ_PySR':>10s}  "
      f"{'PySR/GP':>8s}  {'perf1D/GP':>10s}")
print("-" * 70)
for i, name in enumerate(params):
    sg = sigmas["GP"][i]
    sp = sigmas["perfect_1D"][i]
    if result["pysr_available"]:
        sy = sigmas["PySR"][i]
        print(f"{name:<12s}  {sg:>10.4g}  {sp:>10.4g}  {sy:>10.4g}  "
              f"{sy/sg:>8.3f}  {sp/sg:>10.4f}")
    else:
        print(f"{name:<12s}  {sg:>10.4g}  {sp:>10.4g}  {'n/a':>10s}  "
              f"{'n/a':>8s}  {sp/sg:>10.4f}")

### The σ_perfect_1D ≡ σ_GP check

The `perf1D/GP` column should be ≈ 1.000 for all parameters. Any
deviation indicates a numerical issue (e.g. step-size not converged).
The meaningful ratio is `σ_PySR / σ_GP` — how much constraining power
is lost to PySR fit error.

## 4. Read the forecast table and scorecard

In [ ]:
print(open(result["table_path"]).read())

In [ ]:
print(open(result["scorecard_path"]).read())

## 5. Load and display the corner plot

The corner plot overlays GP / perfect_1D / PySR Fisher ellipses for a
subset of parameters. It is written to `<output_dir>/corner.png`.

In [ ]:
from pathlib import Path

corner_path = result.get("corner_path")
if corner_path and Path(corner_path).exists():
    # In a Jupyter environment this displays inline.
    from IPython.display import Image
    display(Image(filename=str(corner_path)))
else:
    print(f"Corner plot not found at {corner_path}.")

## 6. Dive into the internal Pareto-CSV loading

The helper functions in `forecast.py` are exposed for interactive
inspection:

In [ ]:
from priya_forecast.single_z.forecast import resolve_pareto_csvs

try:
    csv_paths = resolve_pareto_csvs(cfg)
    for param, path in csv_paths.items():
        print(f"  {param:<12s} -> {path}")
except FileNotFoundError as e:
    print(f"[FileNotFoundError] {e}")
    print("\nBundled baseline CSVs are populated once a Stage-C run has been")
    print("vendored. Use pareto_csvs.source='from_refit' if you have your own.")

In [ ]:
# Load one Pareto front directly and inspect it
try:
    from priya_forecast.models.pysr_model import load_pareto_csv
    from priya_forecast.single_z.forecast import _filter_fisher_safe

    example_csv = csv_paths["ns"]
    df = load_pareto_csv(example_csv)
    print("Full Pareto front for 'ns':")
    print(df[["Complexity", "Loss", "Equation"]].to_string(index=False))

    safe = _filter_fisher_safe(df, n_features=3)
    print(f"\nFisher-safe rows: {len(safe)} / {len(df)}")
except (NameError, FileNotFoundError, KeyError):
    print("Pareto CSVs not available; skipping this cell.")

## 7. Running from the CLI

```bash
python scripts/run_pipeline.py \
    --config configs/single_z/example.yaml \
    --mode   forecast_only
```

To run all 13 z-bins and aggregate:

```bash
python scripts/run_batch.py \
    --config configs/single_z/example.yaml \
    --mode   forecast_only
# This calls aggregate_z.aggregate() automatically.
```